![Week 1 Day 4](../../assets/banners/w1d4.png)

# Week 1 Day 4 - Solution: A Third Tool

The Day 4 exercise: add a `list_questions(question_type)` tool, ask which numerical questions are on the paper and what share of the marks they carry, rehearse with the scripted model, then run it for real. Bonus: show the model page 2 of the paper and have it check the marks it reads with `get_question`.

> 💡 Try the exercise yourself first (it's at the end of `week1/day4.ipynb`). Struggling a little is where the learning happens! Then come back and compare.

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import json
import os

import pymupdf
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from PIL import Image

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.models import chat_model, vision_model
from gradewise.schemas import ExamPaper
from gradewise.transcription import image_block

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
        print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(settings.describe_models())
print(datasets.describe())

## Part 1: The exam, the Day 4 tools, and the new one

In [ ]:
EXAM = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in EXAM.questions}
TYPES = sorted({q.question_type for q in EXAM.questions})


@tool
def get_question(question_id: str) -> str:
    """Look up one CS201 midterm question by its ID (e.g. "Q3", "Q7b"): returns its text and max marks."""
    q = QUESTIONS.get(question_id.strip())
    if q is None:
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    return json.dumps({"id": q.id, "text": q.text, "max_marks": q.max_marks})


@tool
def calculate_percentage(marks_scored: float, marks_available: float) -> float:
    """Turn a score into a percentage, rounded to 1 decimal place. Use this instead of doing arithmetic yourself."""
    return round(100 * marks_scored / marks_available, 1)


@tool
def list_questions(question_type: str) -> str:
    """List every question of one type on the CS201 midterm, with its marks, plus the paper's total marks.
    Types: definition, short, long, compare, numerical, mcq."""
    kind = question_type.strip().lower()
    if kind not in TYPES:  # 💡 a helpful error the model can recover from
        return f"No questions of type {question_type!r}. Types on this paper: {', '.join(TYPES)}"
    found = [{"id": q.id, "max_marks": q.max_marks} for q in EXAM.questions if q.question_type == kind]
    return json.dumps({"questions": found, "paper_total_marks": EXAM.total_marks})


tools = [get_question, calculate_percentage, list_questions]

print(list_questions.invoke({"question_type": "numerical"}))
print(list_questions.invoke({"question_type": "essay"}))

## Part 2: The tool loop (from Day 4) and a scripted rehearsal

In [ ]:
def run_tool_loop(model, question: str, tools: list, max_turns: int = 5) -> str:
    by_name = {t.name: t for t in tools}
    model_with_tools = model.bind_tools(tools)
    messages = [
        SystemMessage("You answer questions about the CS201 midterm. Use the tools; never guess marks or do sums."),
        HumanMessage(question),
    ]
    for turn in range(1, max_turns + 1):
        ai = model_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.text
        for call in ai.tool_calls:
            print(f"🔧 turn {turn}: {call['name']}({call['args']})")
            tool_ = by_name.get(call["name"])
            result = (
                tool_.invoke(call) if tool_ else ToolMessage(f"Unknown tool {call['name']!r}", tool_call_id=call["id"])
            )
            print(f"   ↳ {result.content}")
            messages.append(result)
    return f"🙋 Gave up after {max_turns} turns"


class ScriptedModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


question = "Which numerical questions are on the paper, and what percentage of the whole paper are they worth?"

script = [
    AIMessage("", tool_calls=[{"name": "list_questions", "args": {"question_type": "numerical"}, "id": "c1"}]),
    AIMessage(
        "",
        tool_calls=[{"name": "calculate_percentage", "args": {"marks_scored": 10, "marks_available": 50}, "id": "c2"}],
    ),
    AIMessage("Q5, Q7a and Q7b are numerical: 10 of the 50 marks, so 20.0% of the paper."),
]
print("💬", run_tool_loop(ScriptedModel(messages=iter(script)), question, tools))

## Part 3: The real model

A good model calls `list_questions("numerical")`, adds 5 + 2 + 3 = 10 (or, better, passes the numbers to `calculate_percentage`), and answers **20%**.

In [ ]:
print("💬", run_tool_loop(chat_model(temperature=0), question, tools))

## Bonus: vision + tools

Page 2 of the question paper as an image, and an instruction to verify every mark it reads with `get_question`. The model now *reads* with its eyes and *checks* with your tools: the same "trust, but verify" idea GradeWise uses everywhere.

In [ ]:
with pymupdf.open(DATA_DIR / "exams" / "cs_midterm" / "question_paper.pdf") as doc:
    pix = doc[1].get_pixmap(dpi=100, alpha=False)
page2 = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

check_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Read the marks for every question on this page. Then check each one with "
            "get_question (IDs look like Q7a) and report any that disagree.",
        },
        image_block(page2),  # 💡 {"type": "image", "base64": ..., "mime_type": "image/jpeg"}: works on every provider
    ]
)

vision_with_tools = vision_model().bind_tools(tools)
messages = [check_message]
for _ in range(5):  # 💡 the same loop, inlined, because our first message holds an image
    ai = vision_with_tools.invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        break
    for call in ai.tool_calls:
        print(f"🔧 {call['name']}({call['args']})")
        messages.append({t.name: t for t in tools}[call["name"]].invoke(call))

display(Markdown(ai.text))